# Unit 2, Lecture 1: Workflow versus agency

One problem, built three ways. The point is not that the agent is best. The point
is that the simplest shape that works is usually the right one to ship, and you
can prove that with cold numbers rather than opinion.

The problem: **a support ticket comes in, route it to one of five queues**:
billing, technical, account, sales, abuse. That is the whole task.

## Setup

In [1]:
import sys

from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / "src"))

In [2]:
from cse476.lanes import get_client, MODEL, describe
from cse476.triage import (
    SAMPLE_TICKETS, QUEUES, compare,
    triage_workflow, triage_router, triage_agent,
)

print(describe())
client = get_client()
print("queues:", QUEUES)
print()
for name, text in SAMPLE_TICKETS.items():
    print(f"  {name:10} {text}")

Lane: Groq (groq, free)  |  Model: openai/gpt-oss-120b
queues: ['billing', 'technical', 'account', 'sales', 'abuse']

  easy       I was charged twice for my subscription this month.
  ambiguous  Nothing works and I want my money back.
  urgent     Someone has logged into my account and is sending spam from it.
  vague      hello


## Build one: the workflow

No model at all. Fixed keyword rules, same order every time. This is the baseline
everything else is measured against.

In [3]:
for name, text in SAMPLE_TICKETS.items():
    t = triage_workflow(text)
    print(f"{name:10} -> {t.queue:10} ({t.model_calls} calls)  {t.reason}")

easy       -> billing    (0 calls)  Matched a keyword rule for 'billing'.
ambiguous  -> billing    (0 calls)  Matched a keyword rule for 'billing'.
urgent     -> abuse      (0 calls)  Matched a keyword rule for 'abuse'.
vague      -> technical  (0 calls)  No rule matched, defaulted to technical for a human to sort.


It nailed the easy ticket for **zero cost**. Now look at the `ambiguous` one:
"Nothing works and I want my money back."

Is that technical, or billing? The keyword rules grab whichever matches first,
with no understanding of intent. **That is the workflow hitting its ceiling**, and
it is the first honest reason to bring in a model.

## Build two: the router

One model call classifies the ticket into a queue. Then ordinary code takes over.
This is the shape most real "AI routing" systems should be.

In [4]:
for name, text in SAMPLE_TICKETS.items():
    t = triage_router(client, MODEL, text)
    print(f"{name:10} -> {t.queue:10} ({t.model_calls} call)   {t.reason}")

easy       -> billing    (1 call)   Model chose 'billing'.
ambiguous  -> billing    (1 call)   Model chose 'billing'.
urgent     -> abuse      (1 call)   Model chose 'abuse'.
vague      -> account    (1 call)   Model chose 'account'.


The router understands the ambiguous ticket, because it reads intent rather
than keywords. But it still makes exactly **one** decision, from a fixed menu, and
the cost is bounded and known: one call, every time.

### The line that saves the router

```python
queue = choice if choice in QUEUES else "technical"
```

The model was asked for one word from a fixed list. A language model can always
return something off-menu. So we validate against the whitelist and default
anything invalid, exactly the same rule as validating a tool name in Unit 1.
Anything a model produces is untrusted input.

## Build three: the agent

Now the model decides the whole path, using tools to inspect the queues and their
policies, until it is sure. This is the only one of the three that is actually an
agent. It is your `tiny_agent` from Unit 1, pointed at triage.

In [5]:
for name, text in SAMPLE_TICKETS.items():
    t = triage_agent(client, MODEL, text)
    print(f"{name:10} -> {t.queue:10} ({t.model_calls} calls)  {t.reason[:50]}")

easy       -> billing    (3 calls)  QUEUE: billing – the user reports a duplicate char
ambiguous  -> billing    (2 calls)  QUEUE: billing — the user is requesting a refund, 
urgent     -> abuse      (3 calls)  QUEUE: abuse — The issue involves unauthorized acc
vague      -> technical  (5 calls)  Did not decide within 5 steps, defaulted to techni


## The punchline

Put all three on the same ticket, side by side.

In [6]:
ticket = SAMPLE_TICKETS["easy"]   # "I was charged twice..."
print(f"ticket: {ticket}\n")

results = {
    "workflow": triage_workflow(ticket),
    "router":   triage_router(client, MODEL, ticket),
    "agent":    triage_agent(client, MODEL, ticket),
}
print(compare(results))

ticket: I was charged twice for my subscription this month.

shape       calls queue        reason
workflow        0 billing      Matched a keyword rule for 'billing'.
router          1 billing      Model chose 'billing'.
agent           2 billing      QUEUE: billing – The user reports being charged twic


Same queue, every time. But roughly **0, 1, and 3 model calls**.

For a straightforward ticket, the agent paid three times what the router paid for
an identical outcome. The exact numbers vary run to run because the model chooses,
but the direction is stable and it is the whole lesson.

Multiply that gap by a million tickets a month and the choice of shape stops being
a style question and becomes a line item on a budget.

## The test you can apply to anything

Three questions, in order. Stop at the first yes.

1. **Can I write the steps in advance?** → build a workflow.
2. **Is it one decision, then a fixed path?** → build a router.
3. **Do the steps genuinely depend on what I find along the way?** → only now, an agent.

Triage is a question-two problem. That is why the router won.

## Your turn

**1. Add delivery to the list of queues.**

**2. Add appropriate delivery-related keywords such as:**
delivery
shipment
package
tracking
late

**3. Test your program using the given sample tickets.**

**4. Make sure the program prints the correct department for each ticket.**

In [7]:
# your work here


# Support Ticket Classifier

# 1. Add delivery to the list of queues
queues = ["billing", "technical", "account", "sales", "delivery"]


# 2. Add keywords for each department
rules = {
    "billing": ["refund", "charged", "payment"],
    "technical": ["error", "bug", "crash"],
    "account": ["password", "login"],
    "sales": ["price", "upgrade"],
    
    # New delivery queue
    "delivery": ["delivery", "shipment", "package", "tracking", "late"]
}


# Function to classify a ticket
def classify_ticket(ticket):
    ticket = ticket.lower()

    for queue, keywords in rules.items():
        for word in keywords:
            if word in ticket:
                return queue

    return "unknown"


# 3. Test with sample tickets
tickets = [
    "My payment failed",
    "I forgot my password",
    "The application has a bug",
    "Where is my shipment?",
    "My package is late",
    "Can I check the tracking?"
]


# 4. Print the result
for ticket in tickets:
    department = classify_ticket(ticket)
    print(ticket, "→", department)